# Part 1
---

In [1]:
import base64
import json
from typing import Literal

from pydantic import BaseModel
from urllib.parse import urlencode


AppColorScheme = Literal["dark", "light"]
ServiceProviderType = Literal["test", "dev", "custom", "system"]
ServiceProviderOption = bool | int | float | str


class ServiceProviderMeta(BaseModel):
    type: ServiceProviderType
    title: str
    description: str | None = None
    disabled: bool | None = None
    hidden: bool | None = None


class SerializedAppConfig(BaseModel):
    serviceProviderId: str
    serviceProviderMeta: ServiceProviderMeta
    serviceProviderOptions: dict[str, ServiceProviderOption] = {}


def get_query_args(
    compact: bool = True,
    scheme: AppColorScheme | None = None,
    config: SerializedAppConfig | None = None,
) -> str:
    params: dict[str, str] = {}

    if compact:
        params["compact"] = "1"

    if scheme is not None:
        params["scheme"] = scheme
        
    if config is not None:
        params["config"] = _base64url_json(config)

    return f"?{urlencode(params)}" if params else ""


def _base64url_json(value: BaseModel) -> str:
    data = value.model_dump(mode="json", exclude_none=True)
    json_bytes = json.dumps(data, separators=(",", ":")).encode("utf-8")
    return base64.urlsafe_b64encode(json_bytes).decode("ascii").rstrip("=")


In [18]:
config = SerializedAppConfig(
    serviceProviderId="notebook",
    serviceProviderMeta=ServiceProviderMeta(
        type="custom",
        title="Notebook Service",
        description="Configured by Jupyter",
    ),
    serviceProviderOptions={
        "apiUrl": "https://localhost:8008",
        "authType": "none",
    },
)

In [19]:
config.model_dump()

{'serviceProviderId': 'notebook',
 'serviceProviderMeta': {'type': 'custom',
  'title': 'Notebook Service',
  'description': 'Configured by Jupyter',
  'disabled': None,
  'hidden': None},
 'serviceProviderOptions': {'apiUrl': 'https://localhost:8008',
  'authType': 'none'}}

In [20]:
iframe_src = f"https://example.test/eozilla/{get_query_args(config=config)}"

In [21]:
iframe_src

'https://example.test/eozilla/?compact=1&config=eyJzZXJ2aWNlUHJvdmlkZXJJZCI6Im5vdGVib29rIiwic2VydmljZVByb3ZpZGVyTWV0YSI6eyJ0eXBlIjoiY3VzdG9tIiwidGl0bGUiOiJOb3RlYm9vayBTZXJ2aWNlIiwiZGVzY3JpcHRpb24iOiJDb25maWd1cmVkIGJ5IEp1cHl0ZXIifSwic2VydmljZVByb3ZpZGVyT3B0aW9ucyI6eyJhcGlVcmwiOiJodHRwczovL2xvY2FsaG9zdDo4MDA4IiwiYXV0aFR5cGUiOiJub25lIn19'

# Part 2
---

In [22]:
from __future__ import annotations

from contextlib import ExitStack
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from socket import socket
from threading import Thread
from types import ModuleType
from typing import Any
from importlib import resources

from IPython.display import HTML, display


_APP_SERVERS: list[tuple[ThreadingHTTPServer, Thread]] = []
_APP_RESOURCE_STACK = ExitStack()


def _find_free_port() -> int:
    with socket() as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def _serve_directory(directory: Path) -> str:
    directory = directory.resolve()
    port = _find_free_port()

    handler = partial(SimpleHTTPRequestHandler, directory=str(directory))
    server = ThreadingHTTPServer(("127.0.0.1", port), handler)
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()

    _APP_SERVERS.append((server, thread))
    return f"http://127.0.0.1:{port}"


def display_app_iframe(
    build_dir: str | Path,
    *,
    compact: bool = True,
    scheme: AppColorScheme | None = None,
    config: SerializedAppConfig | None = None,
    width: str = "100%",
    height: int = 700,
) -> None:
    base_url = _serve_directory(Path(build_dir))
    src = f"{base_url}/index.html{get_query_args(compact=compact, scheme=scheme, config=config)}"

    display(
        HTML(
            f"""
            <iframe
                src="{src}"
                width="{width}"
                height="{height}"
                style="border: 0; width: {width}; height: {height}px;"
                allow="clipboard-read; clipboard-write"
            ></iframe>
            """
        )
    )


def display_packaged_app_iframe(
    package: str | ModuleType,
    resource_dir: str = "app",
    *,
    compact: bool = True,
    scheme: AppColorScheme | None = None,
    config: SerializedAppConfig | None = None,
    width: str = "100%",
    height: int = 700,
) -> None:
    resource = resources.files(package).joinpath(resource_dir)

    # Keeps extracted package data alive for the notebook session if needed.
    build_dir = _APP_RESOURCE_STACK.enter_context(resources.as_file(resource))

    display_app_iframe(
        build_dir,
        compact=compact,
        scheme=scheme,
        config=config,
        width=width,
        height=height,
    )

In [26]:
display_app_iframe(
    "C:/Users/norma/Projects/eozilla-app/dist",
    scheme="dark",
    config=config,
    height=700
)

127.0.0.1 - - [04/Jun/2026 18:23:13] "GET /index.html?compact=1&scheme=dark&config=eyJzZXJ2aWNlUHJvdmlkZXJJZCI6Im5vdGVib29rIiwic2VydmljZVByb3ZpZGVyTWV0YSI6eyJ0eXBlIjoiY3VzdG9tIiwidGl0bGUiOiJOb3RlYm9vayBTZXJ2aWNlIiwiZGVzY3JpcHRpb24iOiJDb25maWd1cmVkIGJ5IEp1cHl0ZXIifSwic2VydmljZVByb3ZpZGVyT3B0aW9ucyI6eyJhcGlVcmwiOiJodHRwczovL2xvY2FsaG9zdDo4MDA4IiwiYXV0aFR5cGUiOiJub25lIn19 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jun/2026 18:23:13] "GET /assets/index-D1QHV3gI.css HTTP/1.1" 200 -
127.0.0.1 - - [04/Jun/2026 18:23:13] "GET /assets/index-dHo64imR.js HTTP/1.1" 200 -
127.0.0.1 - - [04/Jun/2026 18:23:13] "GET /privacy.md HTTP/1.1" 200 -
